# 04 — Evaluate one model
One notebook, one harness, every model. Change `MODEL_TAG` and run again:

`zeroshot` (untouched base model), `raw1x`, `rawrepeat`, `aug100k`.

All four use the same code, same images, same prompt, same precision, greedy decoding. Results are saved after every 25 images, so if the run is interrupted, running it again resumes where it stopped.

In [ ]:
MODEL_TAG  = "zeroshot"     # "zeroshot" | "raw1x" | "rawrepeat" | "aug100k"
EVAL_SPLIT = "augg"        # "test" for final numbers, "val" for quick checks

In [ ]:
import os, sys, time
os.environ["UNSLOTH_WARN_UNINITIALIZED"] = "0"
import config as C
sys.path.insert(0, str(C.DEEPSEEK_SRC))
import pandas as pd
from tqdm import tqdm
from unsloth import FastVisionModel
from transformers import AutoModel
from ocr_utils import clean_text, corpus_rates, infer_text

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=str(C.BASE_MODEL), load_in_4bit=False, auto_model=AutoModel, trust_remote_code=True)

if MODEL_TAG != "zeroshot":
    adapter_dir = C.MODELS_DIR / f"{MODEL_TAG}_lora"
    assert adapter_dir.exists(), adapter_dir
    model.load_adapter(str(adapter_dir))
    print("Loaded adapter:", adapter_dir)
else:
    print("Zero-shot: no adapter loaded")

FastVisionModel.for_inference(model)
KW = dict(image_size=C.IMAGE_SIZE, base_size=C.BASE_SIZE, crop_mode=False, save_results=False)

In [ ]:
data = pd.read_csv(C.SPLIT_DIR / f"{EVAL_SPLIT}.csv")
#out_csv = C.RESULTS_DIR / f"preds_{MODEL_TAG}_{EVAL_SPLIT}.csv"
# The following is just for test
out_csv = C.RESULTS_DIR / f"preds_new_{MODEL_TAG}_{EVAL_SPLIT}.csv"

done = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame()
finished = set(done["file_name"]) if len(done) else set()
rows = done.to_dict("records") if len(done) else []
print(f"{len(data)} images | already done: {len(finished)}")

for i, r in enumerate(tqdm(data.itertuples(index=False), total=len(data))):
    if r.file_name in finished: continue
    t0 = time.time()
    try:
        pred = infer_text(model, tokenizer, C.RAW_IMG_DIR / r.file_name, C.PROMPT, **KW)
        err = ""
    except Exception as e:
        pred, err = "", str(e)[:100]
    sec = time.time() - t0
    gt = clean_text(r.text); pr = clean_text(pred)
    rows.append({"file_name": r.file_name, "ground_truth": gt, "prediction": pr, "seconds": sec,
                 "empty": int(len(pr) == 0), "runaway": int(len(pr) > 3 * max(len(gt), 1)), "error": err})
    if len(rows) % 25 == 0:
        pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")
pd.DataFrame(rows).to_csv(out_csv, index=False, encoding="utf-8-sig")

res = pd.DataFrame(rows)
cer, wer = corpus_rates(res["ground_truth"].tolist(), res["prediction"].tolist())
print(f"\n{MODEL_TAG} on {EVAL_SPLIT} (N={len(res)}):  CER = {cer:.4f}   WER = {wer:.4f}")
print(f"empty outputs: {res['empty'].sum()} | runaway outputs (>3x reference length): {res['runaway'].sum()} | "
      f"mean seconds/image: {res['seconds'].mean():.2f}")
print("Saved:", out_csv)

In [ ]:
data = pd.read_csv(C.SPLIT_DIR / f"{EVAL_SPLIT}.csv")
out_csv = C.RESULTS_DIR / f"preds_{MODEL_TAG}_{EVAL_SPLIT}.csv"

done = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame()
finished = set(done["file_name"]) if len(done) else set()
rows = done.to_dict("records") if len(done) else []
print(f"{len(data)} images | already done: {len(finished)}")

In [ ]:
# One-time: rebuild eval_summary.csv from any preds_*.csv files already on disk.
# Safe to run anytime -- it recomputes CER/WER from the saved predictions, it doesn't call the model.
import glob
from ocr_utils import corpus_rates

rows = []
for p in sorted(glob.glob(str(C.RESULTS_DIR / "preds_*.csv"))):
    name = os.path.basename(p)[len("preds_"):-len(".csv")]
    tag, split = name.rsplit("_", 1)
    res = pd.read_csv(p, keep_default_na=False)
    cer, wer = corpus_rates(res["ground_truth"].tolist(), res["prediction"].tolist())
    skip_path = C.RESULTS_DIR / f"skip_{tag}.csv"
    n_skip = len(pd.read_csv(skip_path)) if skip_path.exists() else 0
    rows.append({"tag": tag, "split": split, "n": len(res), "cer": cer, "wer": wer,
                "empty": int(res["empty"].sum()), "runaway": int(res["runaway"].sum()),
                "skipped": n_skip, "mean_seconds": res["seconds"].mean()})
    print(f"{tag:10s} {split:5s}  N={len(res):4d}  CER={cer:.4f}  WER={wer:.4f}")

pd.DataFrame(rows).to_csv(C.RESULTS_DIR / "eval_summary.csv", index=False)
print("\nWrote", C.RESULTS_DIR / "eval_summary.csv")

If the zero-shot model gets stuck in a repetition loop on some image and the notebook seems frozen, interrupt the kernel, restart, and run again. It resumes and continues from the last saved image (the stuck image is retried; if it hangs again, note its file name, exclude it consistently for all models, and report how many were excluded).